# Milestone 2 — Transformer-only against Diffusion

**Optional extension:** *Transformer-only vs diffusion-based generation* (and the BERT-conditioned arm).

Three ways to turn the **same ten captions** into pictures:

| Arm | Encoder | How the image is made | Size |
|---|---|---|---|
| **SD 1.5** | CLIP ViT-L/14 | start from noise, denoise ~20–30 times — continuous latents | 512 |
| **aMUSEd** | CLIP ViT-L/14 | chop into discrete tiles, mask them all, fill in the blanks over ~12 rounds | 256 |
| **LDM-BERT** | BERT | latent diffusion, conditioned on BERT instead of CLIP | 256 |

**SD 1.5 against aMUSEd is the cleanest experiment in the milestone**: both use the *same* CLIP
ViT-L/14 text encoder, so the encoder is held fixed and the only variable is the generative mechanism.
aMUSEd has no noise, no noise schedule and no samplers at all — everything the previous notebook spent
its time tuning simply does not exist here.

**LDM-BERT changes the encoder instead**, which is why it is read as a separate axis rather than as a
fourth point on the same line.

### What does not transfer

The fixed-latent protocol. aMUSEd has no starting noise to hold still — its randomness is in which
tokens it commits to first — so the arms share the **prompts and the seed**, not the latents.
Within-arm comparisons were the previous notebook's job; this one compares across mechanisms, where
only the prompts can be held equal.

Each arm is freed before the next loads. 8 GB cannot hold two.

---
## 0. Setup - this notebook is self-contained

Everything it needs is **in this file**. There is no repository to clone and no `src/` to import:
the cells below write the project's modules to disk beside the notebook, then import them
normally. The code is byte-identical to the tested version in the repo.

Writing real files rather than pasting the functions into the notebook's own namespace is
deliberate. `clip_embeddings`, `bert_embeddings` and `gpt2_embeddings` each define `load`,
`encode_text` and `describe`, so flattening them together would silently shadow two of the three.

**Paths.** On Kaggle, everything written goes to `/kaggle/working/` - artefacts to
`/kaggle/working/outputs/` (plots, tables, generated images, the sweep) and milestone 1's data to
`/kaggle/working/data/`. Nothing is written to `/kaggle/input/`, which is read-only.

Attach milestone 1's `data/processed/` as a Kaggle Dataset; the next cell searches all of
`/kaggle/input` for it, however deeply it is nested, and copies it somewhere writable. The copy is
needed because `cfg.ensure_dirs()` creates directories under `DATA_DIR`, and the encoders notebook
saves its GPT-2 embeddings there.

Off Kaggle the same cell does nothing and paths stay relative to the working directory.

In [ ]:
# Install only what is actually missing -- Kaggle ships torch and transformers already.
import importlib.util
import subprocess
import sys
from pathlib import Path

missing = [p for p in ("diffusers", "transformers", "accelerate", "safetensors", "ftfy", "scipy")
           if importlib.util.find_spec(p) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

Path("modules").mkdir(exist_ok=True)
print("ready -- writing modules next")

In [ ]:
# ---- milestone 1 data in, artefacts out ----------------------------------
# Auto-discovery searches the whole of /kaggle/input, so it does not matter how deeply
# the Dataset is nested. Set this only to override it.
DATA_SOURCE = None        # e.g. "/kaggle/input/coco-processed/processed"

import os
import shutil
from pathlib import Path

if Path("/kaggle/input").is_dir():
    os.environ["T2I_OUTPUT_DIR"] = "/kaggle/working/outputs"

    source = (Path(DATA_SOURCE) if DATA_SOURCE else
              next((p.parent for p in sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))),
                   None))
    if source is None:
        found = sorted(p.name for p in Path("/kaggle/input").iterdir())
        raise RuntimeError(
            "No attached Dataset contains processed/pairs.parquet. Attach milestone 1's "
            "data/processed/, or set DATA_SOURCE above. Under /kaggle/input: "
            + (", ".join(found) if found else "nothing attached"))

    # Copied, not read in place: /kaggle/input is read-only, config.ensure_dirs() creates
    # directories under DATA_DIR, and the encoders notebook saves GPT-2 embeddings there.
    work = Path("/kaggle/working/data")
    if not (work / "processed").is_dir():
        print(f"copying {source} ->\n        {work / 'processed'} ...")
        shutil.copytree(source, work / "processed")
    os.environ["T2I_DATA_DIR"] = str(work)

    print("data    :", os.environ["T2I_DATA_DIR"])
    print("outputs :", os.environ["T2I_OUTPUT_DIR"])
    print("contents:", ", ".join(sorted(p.name for p in (work / "processed").iterdir())))
else:
    print("not on Kaggle -- using ./data and ./outputs")

In [ ]:
%%writefile config.py
"""Every tunable value and every path for the milestone-1 pipeline.

Nothing in this file does any work. It is the one place to change the scale of a
run, the models, the caption filters or where artefacts land -- edit a value
here, restart the kernel, re-run the notebook, and nothing else needs touching.

Paths can also be overridden with environment variables, which is how this runs
on Kaggle without editing the file:

    T2I_PROJECT_ROOT   repo root            (default: the parent of src/)
    T2I_DATA_DIR       dataset + artefacts  (default: <root>/data)
    T2I_OUTPUT_DIR     plots, tables, PNGs  (default: <root>/outputs)
"""

import os
from pathlib import Path

import torch

# ---- Paths -------------------------------------------------------------
# Derived from this file's own location, so it does not matter where the kernel
# was started from -- src/config.py -> parents[1] is the repo root.
PROJECT_ROOT = Path(os.environ.get("T2I_PROJECT_ROOT") or Path.cwd())

DATA_DIR = Path(os.environ.get("T2I_DATA_DIR") or PROJECT_ROOT / "data")
ANN_FILE = DATA_DIR / "annotations" / "captions_val2017.json"
IMAGES_DIR = DATA_DIR / "val2017"

PROCESSED_DIR = DATA_DIR / "processed"
IMAGES_512_DIR = PROCESSED_DIR / "images_512"      # 512x512 RGB JPEG cache
EMB_DIR = PROCESSED_DIR / "embeddings"             # the .npy arrays
MANIFEST_PARQUET = PROCESSED_DIR / "pairs.parquet"
MANIFEST_CSV = PROCESSED_DIR / "pairs.csv"
METADATA_FILE = PROCESSED_DIR / "metadata.json"

OUTPUT_DIR = Path(os.environ.get("T2I_OUTPUT_DIR") or PROJECT_ROOT / "outputs")
GENERATIONS_DIR = OUTPUT_DIR / "generations"       # generated PNGs
PLOTS_DIR = OUTPUT_DIR / "plots"                   # every figure, as PNG
TABLES_DIR = OUTPUT_DIR / "tables"                 # every summary table, as JSON

SWEEP_DIR = OUTPUT_DIR / "sweep"                   # one sub-directory per swept config

ALL_DIRS = (PROCESSED_DIR, IMAGES_512_DIR, EMB_DIR, GENERATIONS_DIR, PLOTS_DIR, TABLES_DIR,
            SWEEP_DIR)


def ensure_dirs():
    """Create every output directory. Called once near the top of the notebook."""
    for d in ALL_DIRS:
        d.mkdir(parents=True, exist_ok=True)


# ---- Subset ------------------------------------------------------------
N_IMAGES = 3000             # distinct images kept
CAPTIONS_PER_IMAGE = 2      # captions kept per image -> 3000 x 2 = 6000 pairs
IMAGE_SIZE = 512            # Stable Diffusion v1.x native resolution
SEED = 42                   # single source of randomness for the whole pipeline

# ---- Caption filters ---------------------------------------------------
MIN_WORDS, MAX_WORDS = 3, 60
MIN_CHARS, MAX_CHARS = 10, 250

# ---- Models ------------------------------------------------------------
CLIP_JOINT_MODEL = "openai/clip-vit-base-patch32"    # text + vision towers -> shared 512-d space
CLIP_SD_MODEL = "openai/clip-vit-large-patch14"      # the text encoder SD v1.5 is conditioned on
BERT_MODEL = "google-bert/bert-base-uncased"         # text-only Transformer, for comparison
GPT2_MODEL = "openai-community/gpt2"                 # autoregressive encoder, milestone 2's optional arm
SD_REPO_CANDIDATES = [                               # tried in order; the original runwayml repo is gone
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "sd-legacy/stable-diffusion-v1-5",
    "CompVis/stable-diffusion-v1-4",
]

CLIP_MAX_TOKENS = 77   # fixed by CLIP's positional embedding table
BERT_MAX_TOKENS = 64   # COCO captions are short; 64 covers them with room to spare
GPT2_MAX_TOKENS = 64   # matched to BERT so the encoder comparison varies only the model
BATCH_SIZE = 64

# ---- Generation (section 11; the knobs milestone 2 sweeps) -------------
N_STEPS = 30           # denoising steps
GUIDANCE = 7.5         # classifier-free guidance scale
N_PROMPTS = 5          # test prompts drawn from the curated subset
PROMPT_MIN_WORDS = 8   # prompts long enough to describe a scene, not one object
PROMPT_MAX_WORDS = 18

# ---- Precision ---------------------------------------------------------
# Text encoders in float32: they are small, and float32 keeps the stored embeddings
# free of rounding so cosine similarities are exact. Stable Diffusion in float16 on
# GPU: the UNet is the expensive part and fp16 roughly halves VRAM and step time.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODER_DTYPE = torch.float32
SD_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

# ---- Toggles -----------------------------------------------------------
# The 77x768 sequence tensors are ~710 MB at 6,000 captions. The SD text encoder is
# deterministic in eval mode, so re-running it reproduces them bit-for-bit in well
# under a minute -- cheaper to recompute than to store.
SAVE_SEQ_EMBEDDINGS = False
RUN_SD_GENERATION = True    # section 11 needs ~4 GB of Stable Diffusion weights on first run
N_ALIGN_IMAGES = 500        # images put through CLIP's vision tower for the retrieval check
REUSE_EXISTING = True       # reuse cached embeddings when they match this run's subset
FORCE_RECOMPUTE = False     # set True to ignore every cache and redo all work


# ---- Milestone 2: the sweep ---------------------------------------------
N_FIXED_PROMPTS = 10   # every config generates this same set, from the same latents
LATENT_SEED = SEED     # latents are built once, on CPU, and passed into every run

CFG_GRID = (1.0, 4.0, 7.5, 11.0, 15.0)   # spec sweeps 1-15; 7.5 is the SD 1.5 default
GUIDANCE_RESCALE = 0.7                   # corrects the oversaturation CFG > 12 causes

# Sampler -> step count. The counts differ on purpose: DPMSolver++ and UniPC reach at
# 20 steps what PNDM needs 50 for, so an equal-step comparison would measure nothing
# but the step count.
SAMPLERS = {
    "pndm": 50,      # SD 1.5 default -- the baseline
    "ddim": 50,      # deterministic at eta=0
    "lms": 30,       # multistep; needs scipy
    "euler": 30,
    "euler_a": 30,   # ancestral -- adds noise each step, never converges
    "dpmpp": 20,
    "unipc": 20,
}

# Noise-schedule axes. Swept as a product against one fixed sampler, so the only
# variable is the schedule.
# SD 1.5's shipped scheduler config omits timestep_spacing, so each scheduler class would
# fall back to its own default -- PNDM and DDIM to "leading", the rest to "linspace". The
# guidance and sampler grids pin it so that axis stays still while the schedule grid sweeps it.
BASE_TIMESTEP_SPACING = "leading"   # SD legacy, what v1.5 was trained under

TIMESTEP_SPACINGS = ("leading", "trailing", "linspace")
ZERO_SNR_OPTIONS = (False, True)
KARRAS_OPTIONS = (False, True)
SCHEDULE_SAMPLER = "dpmpp"

DDPM_STEPS = 1000      # the illustration, not a sweep arm
DDPM_N_PROMPTS = 2

# Batch 4, not the 8 an unshared 16 GB T4 allows: this targets an 8 GB RTX 5060 Laptop
# GPU that is also driving the display. SD 1.5 in fp16 is ~2.5 GB of weights and roughly
# 0.6 GB of activations per 512x512 image.
GEN_BATCH_SIZE = 4

In [ ]:
%%writefile utils.py
"""Small helpers every section uses: seeding, path printing, and saving figures
and summary tables to `outputs/`.

Kept out of `config.py` on purpose -- config holds values you edit, this holds
code you call.
"""

import json
import os
import random
from pathlib import Path

import numpy as np
import torch

import config as cfg


def set_seed(seed=None):
    """Reseed every RNG the pipeline touches.

    Called once at the start and again before generation, so the subset, the
    sampled examples and the generated images are identical on every re-run.
    """
    seed = cfg.SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Deterministic convolution kernels: slightly slower, but two runs produce
    # identical tensors.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed


def rel(path):
    """`path` relative to the project root, as a string, for printing.

    Falls back to a computed relative path instead of raising when the path sits
    outside the root -- which happens on Kaggle, where the data and the code live
    on different mounts -- and to the absolute path when even that is impossible,
    as on Windows when the two are on different drives.
    """
    p = Path(path)
    try:
        return p.relative_to(cfg.PROJECT_ROOT).as_posix()
    except ValueError:
        pass
    try:
        return Path(os.path.relpath(p, cfg.PROJECT_ROOT)).as_posix()
    except ValueError:
        return p.as_posix()


def save_fig(fig, name):
    """Write a figure to `outputs/plots/` and return its path for printing."""
    cfg.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.PLOTS_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return rel(path)


def save_table(name, data):
    """Write a summary table to `outputs/tables/` as JSON.

    `default=float` converts numpy scalars, which json cannot serialise.
    """
    cfg.TABLES_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.TABLES_DIR / f"{name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=float)
    return rel(path)

In [ ]:
%%writefile modules/__init__.py
"""Pipeline stages, written by the setup cells above."""

In [ ]:
%%writefile modules/arms.py
"""The two comparison arms: aMUSEd and LDM-BERT.

**aMUSEd** answers "transformer-only against diffusion" as cleanly as it can be
answered. It uses the same CLIP ViT-L/14 text encoder as SD 1.5, so the encoder is
held fixed and the only thing that changes is how the image is made: SD denoises
continuous latents over ~30 steps, aMUSEd chops the image into discrete tiles, masks
them all, and fills in the blanks over ~12 rounds -- BERT's masked-prediction trick
applied to pictures. No noise, no noise schedule, no samplers.

That also means **the fixed-latent protocol does not transfer**. There is no starting
noise to hold still; aMUSEd's randomness is in which tokens it commits to first. The
arms share the prompts and the seed, not the latents.

**LDM-BERT** is the BERT-conditioned arm, at 256px.

`run_arm` passes only the keyword arguments a pipeline actually declares. The three
pipelines do not share a call signature -- `guidance_rescale` is meaningless to a
masked transformer -- and LDM's signature ends in `**kwargs`, which would swallow a
wrong argument silently rather than raising. Named parameters only, for all of them.
"""

import inspect

from diffusers import AmusedPipeline, LDMTextToImagePipeline

import config as cfg
from . import generation, sd_pipeline

ARMS = {
    "amused": {
        "repo": "amused/amused-256",
        "pipeline": AmusedPipeline,
        "size": 256,
        "kind": "masked transformer",
        "encoder": "CLIP ViT-L/14 -- the same one SD 1.5 uses",
    },
    "ldm_bert": {
        "repo": "CompVis/ldm-text2im-large-256",
        "pipeline": LDMTextToImagePipeline,
        "size": 256,
        "kind": "latent diffusion",
        "encoder": "BERT",
    },
}


def load_arm(name, device=None, dtype=None, previous=None):
    """Load one arm, freeing `previous` first. Returns `(pipe, spec)`.

    8 GB cannot hold two of these at once, so the previous arm goes before this one
    arrives rather than after.
    """
    spec = ARMS[name]
    device = cfg.DEVICE if device is None else device
    dtype = cfg.SD_DTYPE if dtype is None else dtype

    if previous is not None:
        sd_pipeline.free(previous)

    pipe = spec["pipeline"].from_pretrained(spec["repo"], torch_dtype=dtype).to(device)
    pipe.set_progress_bar_config(disable=True)
    sd_pipeline.enable_memory_savers(pipe)
    return pipe, spec


def load_amused(**kwargs):
    return load_arm("amused", **kwargs)


def load_ldm_bert(**kwargs):
    return load_arm("ldm_bert", **kwargs)


def accepted_kwargs(pipe, kwargs):
    """Only the arguments this pipeline names explicitly -- `**kwargs` does not count."""
    params = inspect.signature(pipe.__call__).parameters
    named = {k for k, p in params.items()
             if p.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)}
    return {k: v for k, v in kwargs.items() if k in named}


def run_arm(pipe, prompts, batch_size=None, **kwargs):
    """Generate one image per prompt, passing only what this pipeline accepts.

    Batched for the same reason `generation.generate` is: classifier-free guidance
    doubles the effective UNet batch, so ten 512px prompts in one call is twenty, which
    does not fit alongside the weights on an 8 GB card.
    """
    batch_size = cfg.GEN_BATCH_SIZE if batch_size is None else batch_size
    accepted = accepted_kwargs(pipe, kwargs)
    prompts = list(prompts)

    images = []
    for part in generation.batches(len(prompts), batch_size):
        images.extend(pipe(prompt=prompts[part], **accepted).images)
    return images

In [ ]:
%%writefile modules/clip_embeddings.py
"""CLIP text and image encoders.

Two checkpoints, doing two different jobs:

  `clip-vit-base-patch32`   pooled 512-d vectors in a space shared by text AND
                            images, so a caption vector and a photo vector are
                            directly comparable. This is what makes the
                            text -> image retrieval check possible.
  `clip-vit-large-patch14`  the exact text encoder Stable Diffusion v1.5 is
                            conditioned on. Its 77 x 768 per-token states are the
                            same *kind* of tensor the UNet cross-attends to -- no
                            substitutes, because a pretrained UNet only understands
                            the space it was trained in.

**`encode_sequence` here is for analysis, not for conditioning.** It passes an
attention mask, which the pipeline deliberately does not; see the warning on that
function. To build `prompt_embeds`, use `conditioning.encode_prompts`.

The conditioning tensor is one vector per token, not one per sentence, because
the generator reads the prompt token by token. That is how "a *red* bus" colours
the bus instead of tinting the whole picture red.
"""

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPModel, CLIPTextModel, CLIPTokenizer

import config as cfg


def pooled(out):
    """Unwrap `get_text_features` / `get_image_features` across transformers versions.

    transformers >= 5 returns a pooling output object; earlier versions return a
    bare tensor. `pooler_output` is ALREADY projected into the joint space --
    applying `model.text_projection` on top of it would be wrong.
    """
    return out.pooler_output if hasattr(out, "pooler_output") else out


# ---------------------------------------------------------------------------
# Joint text + vision model
# ---------------------------------------------------------------------------
def load_joint(model_name=None, device=None, dtype=None):
    """Load the joint-space CLIP. Returns `(tokenizer, model, image_processor)`."""
    model_name = cfg.CLIP_JOINT_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = CLIPTokenizer.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    processor = CLIPImageProcessor.from_pretrained(model_name)
    return tokenizer, model, processor


def token_lengths(texts, tokenizer):
    """Untruncated token count per caption.

    CLIP's positional table stops at 77 tokens, so anything longer is silently
    truncated and the model never sees the end of the caption. Measuring this is
    how we know the word-count filter is already keeping us under the ceiling.
    """
    return np.array([len(tokenizer(t, truncation=False)["input_ids"]) for t in texts])


@torch.no_grad()
def encode_text(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Pooled 512-d vectors in CLIP's shared text/image space. Returns (N, 512) float32."""
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.CLIP_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="CLIP joint-space"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        out.append(pooled(model.get_text_features(**toks)).float().cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def encode_images(images, processor, model, batch_size=32, device=None, progress=True):
    """Pooled 512-d image vectors in the same space as `encode_text`.

    `images` may be paths or already-open PIL images.
    """
    device = cfg.DEVICE if device is None else device
    out = []
    steps = range(0, len(images), batch_size)
    for i in tqdm(steps, desc="CLIP image encode", disable=not progress):
        batch = [
            img if isinstance(img, Image.Image) else Image.open(img)
            for img in images[i:i + batch_size]
        ]
        batch = [img.convert("RGB") for img in batch]
        px = processor(images=batch, return_tensors="pt").to(device)
        out.append(pooled(model.get_image_features(**px)).float().cpu())
    return torch.cat(out).numpy()


# ---------------------------------------------------------------------------
# Stable Diffusion conditioning encoder
# ---------------------------------------------------------------------------
def load_sd_text_encoder(model_name=None, device=None, dtype=None):
    """Load the text encoder SD v1.5 is conditioned on. Returns `(tokenizer, model)`."""
    model_name = cfg.CLIP_SD_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = CLIPTokenizer.from_pretrained(model_name)
    model = CLIPTextModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    return tokenizer, model


@torch.no_grad()
def encode_sequence(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Per-token hidden states (N, 77, 768) plus the pooled EOS vector (N, 768).

    **Do not feed the sequence tensor to a pipeline as `prompt_embeds`.** This passes
    an attention mask; Stable Diffusion's own `encode_prompt` does not. CLIP's text
    tower is causal, so the mask leaves every real token bit-identical -- the pooled
    vector this returns is unaffected, which is why it is fine for the analysis it was
    written for -- but it moves every padded position by ~30 on a scale of 33, and the
    UNet cross-attends to all 77. Images come out different and measurably worse
    (CLIP score 0.23 against 0.34) with nothing raising an error.

    Use `conditioning.encode_prompts` for conditioning.

    The sequence tensor comes back as float16 -- it is ~710 MB at 6,000 captions
    in float32, and halving it costs nothing we can measure downstream.
    """
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.CLIP_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    seq_out, pool_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="CLIP sequence"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        out = model(**toks)
        seq_out.append(out.last_hidden_state.to(torch.float16).cpu())
        pool_out.append(out.pooler_output.float().cpu())
    return torch.cat(seq_out).numpy(), torch.cat(pool_out).numpy()


def describe(model):
    """Parameter and layer counts, for the notebook to print after loading."""
    config = model.config
    out = {"parameters_millions": sum(p.numel() for p in model.parameters()) / 1e6}
    if hasattr(config, "text_config"):
        out["text_hidden_size"] = config.text_config.hidden_size
        out["vision_hidden_size"] = config.vision_config.hidden_size
        out["joint_projection_dim"] = config.projection_dim
    else:
        out["hidden_size"] = config.hidden_size
        out["layers"] = config.num_hidden_layers
    return out

In [ ]:
%%writefile modules/generation.py
"""One swept configuration, and the call that renders it.

`GenConfig` is frozen and its `scheduler_kwargs` is a tuple of pairs rather than a
dict, so a config is hashable -- the sweep puts them in sets and uses `label` as a
directory name, and a mutable config would let two runs share an identity.

`generate` is the only place the *sweep* calls the pipeline, and it is called with
`prompt_embeds=` and never `prompt=`: that is the milestone's integration
requirement, enforced in one place rather than trusted across a notebook. Two places
use `prompt=` on purpose and say so -- the integration check, which exists to compare
the two paths, and the cross-arm comparison, where each arm brings its own encoder.

Batching exists for the 8 GB budget. Latents are sliced alongside the embeddings so
prompt *i* keeps its own noise no matter where the batch boundary falls.
"""

from dataclasses import dataclass, field

import torch

import config as cfg
from . import sd_pipeline

# Flags read better as a word than as `key=True` in a directory name.
_LABEL_FLAGS = {
    "use_karras_sigmas": "karras",
    "rescale_betas_zero_snr": "zerosnr",
}


@dataclass(frozen=True)
class GenConfig:
    """One point in the sweep: a sampler, a step count, a guidance scale, a schedule."""

    sampler: str
    steps: int
    guidance: float
    guidance_rescale: float = 0.0
    scheduler_kwargs: tuple = field(default=())

    @property
    def label(self):
        """A unique, filesystem-safe identifier, e.g. `dpmpp_s20_g7.5_trailing_karras`."""
        parts = [self.sampler, f"s{self.steps}", f"g{self.guidance:g}"]
        if self.guidance_rescale:
            parts.append(f"r{self.guidance_rescale:g}")
        for key, value in self.scheduler_kwargs:
            if value is True:
                parts.append(_LABEL_FLAGS.get(key, key))
            elif value is not False and value is not None:
                parts.append(str(value))
        return "_".join(parts)


def batches(n, size):
    """`n` indices as consecutive slices of at most `size`."""
    return [slice(i, min(i + size, n)) for i in range(0, n, size)]


def prompt_generators(n, seed=None):
    """One seeded CPU generator per prompt.

    Pre-generated latents pin where a run *starts*; they do nothing about the noise
    ancestral samplers (Euler-a) and DDPM draw inside the denoising loop, which
    otherwise comes from the global device RNG and so depends on everything that ran
    before it in the kernel. One generator per prompt rather than one per batch, so a
    prompt's noise does not change when the batch size does.
    """
    seed = cfg.LATENT_SEED if seed is None else seed
    return [torch.Generator(device="cpu").manual_seed(seed + i) for i in range(n)]


def generate(pipe, prompt_embeds, negative_embeds, latents, config, base_config=None,
             batch_size=None, seed=None, **pipe_kwargs):
    """Render one image per row of `prompt_embeds`. Returns a list of PIL images."""
    batch_size = cfg.GEN_BATCH_SIZE if batch_size is None else batch_size
    sd_pipeline.set_scheduler(pipe, config.sampler, base_config=base_config,
                              **dict(config.scheduler_kwargs))

    device, dtype = pipe.device, pipe.dtype
    generators = prompt_generators(len(prompt_embeds), seed)
    images = []
    for part in batches(len(prompt_embeds), batch_size):
        out = pipe(
            prompt_embeds=prompt_embeds[part].to(device=device, dtype=dtype),
            negative_prompt_embeds=negative_embeds[part].to(device=device, dtype=dtype),
            latents=latents[part].to(device=device, dtype=dtype),
            generator=generators[part],
            num_inference_steps=config.steps,
            guidance_scale=config.guidance,
            guidance_rescale=config.guidance_rescale,
            **pipe_kwargs,
        )
        images.extend(out.images)
    return images

In [ ]:
%%writefile modules/prompts.py
"""The fixed prompt set every swept config generates.

A sweep is only a sweep if the prompts never move. Every config in milestone 2 --
each guidance scale, each sampler, each noise schedule -- renders this same set
from the same latents, so a difference in CLIP score is a difference in the knob
and nothing else.

Selection is drawn from the curated manifest rather than hand-written, so the
prompts are real COCO captions the encoders have been measured on, and it is
seeded, so the set is identical on every machine and every re-run.
"""

import config as cfg


def fixed_prompts(manifest, n=None, min_words=None, max_words=None, seed=None):
    """Sample `n` captions in the `[min_words, max_words]` band. Returns manifest rows.

    The word band matters: a three-word caption underdetermines the image, so every
    config scores well and the sweep separates nothing. Eighteen words is where CLIP's
    77-token limit starts to bite.
    """
    n = cfg.N_FIXED_PROMPTS if n is None else n
    min_words = cfg.PROMPT_MIN_WORDS if min_words is None else min_words
    max_words = cfg.PROMPT_MAX_WORDS if max_words is None else max_words
    seed = cfg.SEED if seed is None else seed

    eligible = manifest[manifest["n_words"].between(min_words, max_words)]
    if len(eligible) < n:
        raise ValueError(
            f"{n} prompts requested but only {len(eligible)} captions fall in "
            f"{min_words}-{max_words} words"
        )

    # Sorted after sampling so the row order is the manifest's, not the sampler's --
    # two runs that pick the same captions also pair them with the same latents.
    return eligible.sample(n=n, random_state=seed).sort_values("pair_idx").reset_index(drop=True)

In [ ]:
%%writefile modules/recording.py
"""Turning a run's images into the artefacts the write-up uses.

A sweep's output is only readable as a picture: fifty PNGs in a folder say nothing,
one grid per config says everything. `comparison_grid` is the baseline-vs-best
figure -- one labelled row per config, the same prompts down the columns, so the
knob's effect is the only thing that changes along a column.

Labels are slugified because they come from captions and config names, and a COCO
caption contains characters Windows will not accept in a filename.
"""

import math
import re

from PIL import Image, ImageDraw, ImageFont

import config as cfg
from utils import rel

LABEL_BAND = 20
_UNSAFE = re.compile(r"[^A-Za-z0-9._-]+")


def slugify(text, max_length=60):
    """A filename-safe version of `text`."""
    return _UNSAFE.sub("_", str(text)).strip("_")[:max_length] or "untitled"


def _layout(n, rows, cols):
    if rows is None and cols is None:
        cols = math.ceil(math.sqrt(n))
    if cols is None:
        cols = math.ceil(n / rows)
    if rows is None:
        rows = math.ceil(n / cols)
    return rows, cols


def image_grid(images, rows=None, cols=None, pad=4, background=(255, 255, 255)):
    """Tile `images` into one picture, padded between and around the tiles."""
    if not images:
        raise ValueError("image_grid needs at least one image")

    rows, cols = _layout(len(images), rows, cols)
    w, h = images[0].size
    canvas = Image.new("RGB", (cols * w + (cols + 1) * pad, rows * h + (rows + 1) * pad), background)

    for i, img in enumerate(images):
        r, c = divmod(i, cols)
        canvas.paste(img, (pad + c * (w + pad), pad + r * (h + pad)))
    return canvas


def comparison_grid(rows, pad=4, background=(255, 255, 255)):
    """One labelled row per entry of `rows`, a `{label: [images]}` mapping."""
    if not rows:
        raise ValueError("comparison_grid needs at least one row")

    strips = {label: image_grid(images, rows=1, cols=len(images), pad=pad)
              for label, images in rows.items()}
    width = max(s.width for s in strips.values())
    canvas = Image.new("RGB", (width, sum(s.height + LABEL_BAND for s in strips.values())),
                       background)

    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default()
    y = 0
    for label, strip in strips.items():
        draw.text((pad + 2, y + 4), str(label), fill=(0, 0, 0), font=font)
        canvas.paste(strip, (0, y + LABEL_BAND))
        y += LABEL_BAND + strip.height
    return canvas


def save_run(images, labels, name, out_dir=None, rows=None, cols=None, pad=4):
    """Write every image plus a `grid.png` into `<out_dir>/<name>/`.

    Returns the written paths, relative to the project root, for the notebook to print.
    """
    directory = (cfg.SWEEP_DIR if out_dir is None else out_dir) / name
    directory.mkdir(parents=True, exist_ok=True)

    written = []
    for i, (img, label) in enumerate(zip(images, labels)):
        path = directory / f"{i:02d}_{slugify(label)}.png"
        img.save(path)
        written.append(path)

    grid_path = directory / "grid.png"
    image_grid(images, rows=rows, cols=cols, pad=pad).save(grid_path)
    written.append(grid_path)

    return [rel(p) for p in written]

In [ ]:
%%writefile modules/sanity_checks.py
"""Checks that the embeddings are **meaningful**, not merely well-formed.

Matching shapes prove nothing on their own -- a bug that encoded the same caption
6,000 times would pass a shape check. Four checks, cheapest first:

  1. `embedding_health`  expected dimensions, no NaN/Inf, sane vector norms.
  2. `retrieval`         the real test: does each caption retrieve *its own* image
                         out of 500? Chance is 1/500 = 0.2%.
  3. `sibling_gap`       the two captions of one image describe the same scene, so
                         they should land closer together than two random
                         captions. Needs no image tower, so BERT can take part --
                         which is what makes the CLIP-vs-BERT comparison possible.
  4. `top_neighbours`    the qualitative read: what does each encoder think is
                         similar?

These return numbers. Plotting stays in the notebook, where the figures appear.
"""

import numpy as np
import torch
import torch.nn.functional as F

import config as cfg


def expected_shapes(n_rows, save_seq_embeddings=None):
    """The shape every array should have, given the manifest row count."""
    save_seq = cfg.SAVE_SEQ_EMBEDDINGS if save_seq_embeddings is None else save_seq_embeddings
    expected = {
        "clip_text_joint": (n_rows, 512),
        "clip_sd_pooled": (n_rows, 768),
        "bert_cls": (n_rows, 768),
        "bert_mean": (n_rows, 768),
    }
    if save_seq:
        expected["clip_sd_sequence"] = (n_rows, cfg.CLIP_MAX_TOKENS, 768)
    return expected


def embedding_health(arrays, expected):
    """Shape, finiteness and norm statistics per array.

    Returns `(report, all_ok)`.
    """
    report, all_ok = {}, True
    for name, array in arrays.items():
        a32 = array.astype(np.float32)
        shape_ok = array.shape == expected[name]
        finite = bool(np.isfinite(a32).all())
        norms = np.linalg.norm(a32.reshape(len(a32), -1), axis=1)
        all_ok &= shape_ok and finite
        report[name] = {
            "shape": list(array.shape),
            "expected": list(expected[name]),
            "dtype": str(array.dtype),
            "shape_ok": shape_ok,
            "all_finite": finite,
            "norm_mean": float(norms.mean()),
            "norm_std": float(norms.std()),
        }
    return report, bool(all_ok)


def retrieval(text_emb, image_emb):
    """Text -> image retrieval over a square set of caption/image pairs.

    Retrieval needs exactly one correct answer per query, so the caller passes the
    first caption of each image only -- otherwise two queries compete for the same
    target.

    Returns a dict with the metrics plus the raw `sim`, `ranks`, `matched` and
    `mismatched` arrays, so the notebook can plot them without recomputing.
    """
    txt = F.normalize(torch.from_numpy(text_emb.astype(np.float32)), dim=-1)
    img = F.normalize(torch.from_numpy(image_emb.astype(np.float32)), dim=-1)
    sim = (txt @ img.T).numpy()                   # (n, n): rows = captions, cols = images

    n = len(sim)
    ranks = (-sim).argsort(axis=1)
    rank_of_gt = np.array([int(np.where(ranks[i] == i)[0][0]) for i in range(n)])
    r1, r5, r10 = (float((rank_of_gt < k).mean()) for k in (1, 5, 10))
    chance = 1.0 / n

    gt = np.arange(n)
    matched, mismatched = sim[gt, gt], sim[~np.eye(n, dtype=bool)]

    return {
        "n_queries": int(n),
        "recall_at_1": r1, "recall_at_5": r5, "recall_at_10": r10,
        "chance": chance,
        "times_better_than_chance": float(r1 / chance),
        "median_rank_of_correct_image": int(np.median(rank_of_gt)) + 1,
        "cosine_matched_mean": float(matched.mean()),
        "cosine_matched_std": float(matched.std()),
        "cosine_mismatched_mean": float(mismatched.mean()),
        "cosine_mismatched_std": float(mismatched.std()),
        "separation": float(matched.mean() - mismatched.mean()),
        # raw arrays, for plotting
        "sim": sim, "ranks": ranks, "matched": matched, "mismatched": mismatched,
    }


def sibling_gap(encoders, pair_a, pair_b, rng=None):
    """Sibling-caption agreement per encoder.

    A larger gap means the encoder places two descriptions of one scene closer
    together than two unrelated captions -- it is encoding meaning rather than
    surface word overlap. A high sibling cosine with an equally high random cosine
    means the opposite: everything is mapped into a narrow cone, so the rankings
    carry little information. Watch for that on the [CLS] row.

    Returns `{label: {"sibling_cos": .., "random_cos": .., "gap": ..}}`.
    """
    rng = np.random.default_rng(cfg.SEED) if rng is None else rng
    shuffled = rng.permutation(len(pair_b))

    scores = {}
    for label, emb in encoders.items():
        v = F.normalize(torch.from_numpy(emb.astype(np.float32)), dim=-1)
        sib = (v[pair_a] * v[pair_b]).sum(-1).numpy()
        rnd = (v[pair_a] * v[pair_b][shuffled]).sum(-1).numpy()
        scores[label] = {
            "sibling_cos": float(sib.mean()),
            "random_cos": float(rnd.mean()),
            "gap": float(sib.mean() - rnd.mean()),
        }
    return scores


def top_neighbours(emb, query_idx, k=5):
    """The `k` captions nearest to `query_idx` in this encoder's space.

    Returns `(indices, similarities)`. The query itself is excluded.
    """
    v = F.normalize(torch.from_numpy(emb.astype(np.float32)), dim=-1)
    sims = (v[query_idx] @ v.T).numpy()
    sims[query_idx] = -np.inf
    idx = sims.argsort()[::-1][:k]
    return idx, sims[idx]


def clip_scores(prompts, generated_images, real_images, tokenizer, model, processor, device=None):
    """CLIP cosine similarity for a batch of generation runs.

    Returns `(text_vs_generated, text_vs_real, generated_vs_real)`, each (N,).

    Every number is a cosine in CLIP's joint space. Text-image scores sit in a
    narrow band in practice -- roughly 0.25-0.35 for a caption and an image that
    genuinely match, around 0.1 for an unrelated pair. `generated_vs_real` is on a
    different scale entirely and runs much higher, because two images share far
    more structure with each other than a sentence ever shares with an image;
    compare it across rows, never against the first two.
    """
    from .clip_embeddings import encode_images, pooled   # local import: avoids a cycle at import time

    device = cfg.DEVICE if device is None else device
    with torch.no_grad():
        toks = tokenizer(prompts, padding="max_length", max_length=cfg.CLIP_MAX_TOKENS,
                         truncation=True, return_tensors="pt").to(device)
        t_emb = F.normalize(pooled(model.get_text_features(**toks)).float().cpu(), dim=-1)

    g_emb = F.normalize(torch.from_numpy(
        encode_images(generated_images, processor, model, device=device, progress=False)), dim=-1)
    r_emb = F.normalize(torch.from_numpy(
        encode_images(real_images, processor, model, device=device, progress=False)), dim=-1)

    return (
        (t_emb * g_emb).sum(-1).numpy(),
        (t_emb * r_emb).sum(-1).numpy(),
        (g_emb * r_emb).sum(-1).numpy(),
    )

In [ ]:
%%writefile modules/sd_pipeline.py
"""Loading Stable Diffusion 1.5 and swapping its sampler without losing its training.

Two things here are easy to get wrong.

**The scheduler swap.** `DDIMScheduler()` constructs a scheduler with *library*
defaults, which are not SD 1.5's -- `beta_schedule` would silently become `linear`
where SD 1.5 trained on `scaled_linear`, and every image would come out wrong for
a reason nothing reports. The swap must go through `from_config`. It must also read
the *original* config each time: swapping twice from `pipe.scheduler.config`
compounds the first swap's overrides into the second.

**What each sampler accepts.** The spec's noise-schedule axes are not universal --
`EulerAncestralDiscreteScheduler` has no `use_karras_sigmas`, `PNDMScheduler` has
no `rescale_betas_zero_snr`. Passing one anyway is a TypeError mid-sweep, so the
accepted set is read from each class's signature rather than hardcoded.

Memory: this targets an 8 GB laptop GPU, so slicing is on by default and `free()`
exists to be called between arms.
"""

import gc
import inspect

import torch
from diffusers import (
    DDIMScheduler,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    LMSDiscreteScheduler,
    PNDMScheduler,
    StableDiffusionPipeline,
    UniPCMultistepScheduler,
)

import config as cfg

SCHEDULER_CLASSES = {
    "pndm": PNDMScheduler,
    "ddim": DDIMScheduler,
    "lms": LMSDiscreteScheduler,
    "euler": EulerDiscreteScheduler,
    "euler_a": EulerAncestralDiscreteScheduler,
    "dpmpp": DPMSolverMultistepScheduler,
    "unipc": UniPCMultistepScheduler,
    # Not in cfg.SAMPLERS on purpose: ~1000 steps is 100x the cost of DPMSolver++ for a
    # worse image, so DDPM is a two-prompt illustration rather than a sweep arm.
    "ddpm": DDPMScheduler,
}

SCHEDULER_DEFAULTS = {
    "dpmpp": {"algorithm_type": "dpmsolver++"},
}


def _accepted(scheduler_cls, kwargs):
    """Drop the kwargs this scheduler class cannot take."""
    params = inspect.signature(scheduler_cls.__init__).parameters
    if any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values()):
        return dict(kwargs)
    return {k: v for k, v in kwargs.items() if k in params}


def scheduler_kwargs(name, scheduler_cls=None, **overrides):
    """This sampler's defaults merged with `overrides`, filtered to what it accepts."""
    cls = SCHEDULER_CLASSES[name] if scheduler_cls is None else scheduler_cls
    return _accepted(cls, {**SCHEDULER_DEFAULTS.get(name, {}), **overrides})


def set_scheduler(pipe, name, base_config=None, scheduler_classes=None, **overrides):
    """Swap `pipe.scheduler` to `name`, rebuilt from SD 1.5's original config.

    Pass `base_config` -- the dict `load_sd` captured at load time -- so a second
    swap does not inherit the first one's overrides.
    """
    classes = SCHEDULER_CLASSES if scheduler_classes is None else scheduler_classes
    cls = classes[name]
    base = dict(pipe.scheduler.config) if base_config is None else dict(base_config)
    pipe.scheduler = cls.from_config(base, **scheduler_kwargs(name, scheduler_cls=cls, **overrides))
    return pipe


def enable_memory_savers(pipe):
    """Turn on the slicing an 8 GB card wants, using whichever API this version has.

    diffusers 0.42 removed the pipeline-level `enable_vae_slicing()` wrapper and left
    the method on the VAE itself, so both are probed rather than assumed.
    """
    if hasattr(pipe, "enable_attention_slicing"):
        pipe.enable_attention_slicing()
    vae = getattr(pipe, "vae", None)
    if hasattr(vae, "enable_slicing"):
        vae.enable_slicing()
    return pipe


def load_sd(repo_candidates=None, device=None, dtype=None, slicing=True):
    """Load SD 1.5. Returns `(pipe, repo_id, base_scheduler_config)`.

    The candidates are tried in order because the original `runwayml` repo was
    withdrawn; `base_scheduler_config` is captured before anything touches the
    scheduler and is what every later swap rebuilds from.
    """
    repo_candidates = cfg.SD_REPO_CANDIDATES if repo_candidates is None else repo_candidates
    device = cfg.DEVICE if device is None else device
    dtype = cfg.SD_DTYPE if dtype is None else dtype

    errors = {}
    for repo in repo_candidates:
        try:
            pipe = StableDiffusionPipeline.from_pretrained(
                repo, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False,
            )
        except Exception as exc:
            errors[repo] = f"{type(exc).__name__}: {exc}"
            continue

        pipe = pipe.to(device)
        pipe.set_progress_bar_config(disable=True)
        if slicing and device.type == "cuda":
            enable_memory_savers(pipe)
        return pipe, repo, dict(pipe.scheduler.config)

    raise RuntimeError("no Stable Diffusion repo could be loaded: " + repr(errors))


def free(*objects):
    """Drop references and return the VRAM, so the next arm has room to load."""
    for obj in objects:
        if hasattr(obj, "to"):
            try:
                obj.to("cpu")
            except Exception:
                pass
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def vram_report(device=None):
    """Allocated / reserved / total VRAM in GB, for the notebook to print."""
    device = cfg.DEVICE if device is None else device
    if device.type != "cuda":
        return {"device": str(device)}
    props = torch.cuda.get_device_properties(device)
    gb = 1024 ** 3
    return {
        "device": props.name,
        "total_gb": round(props.total_memory / gb, 2),
        "allocated_gb": round(torch.cuda.memory_allocated(device) / gb, 2),
        "reserved_gb": round(torch.cuda.memory_reserved(device) / gb, 2),
        "peak_gb": round(torch.cuda.max_memory_allocated(device) / gb, 2),
    }

In [ ]:
%%writefile modules/storage.py
"""Everything that touches disk: the manifest, the .npy arrays, metadata, and the
cache logic that lets a re-run skip work it has already done.

The contract these functions enforce: **row *i* of every .npy file is the caption
at `pair_idx == i` in `pairs.parquet`.** One join key, checked before anything is
written and verified by a round-trip read afterwards.

    data/processed/
    |-- pairs.parquet            the curated manifest
    |-- pairs.csv                the same, human-readable
    |-- metadata.json            provenance: models, config, shapes, versions, timings
    |-- images_512/              512x512 RGB JPEG cache
    `-- embeddings/
        |-- clip_text_joint.npy  (N, 512)  float32   shared text/image space
        |-- clip_sd_pooled.npy   (N, 768)  float32   SD v1.5 text encoder, pooled
        |-- bert_cls.npy         (N, 768)  float32   [CLS] token
        `-- bert_mean.npy        (N, 768)  float32   mean-pooled

Figures and summary tables go somewhere separate, under `outputs/`, because they
describe a *run* rather than the dataset itself.

Every path comes from `config.py` -- change `EMB_DIR` there and everything here
follows.
"""

import hashlib
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import config as cfg
from utils import rel

MANIFEST_COLUMNS = [
    "pair_idx", "image_id", "ann_id", "cap_rank", "file_name",
    "caption_raw", "caption", "n_words", "n_chars",
    "height", "width", "image_path", "image_512_path",
]

# One row per retrieval image rather than one per pair, so it is exempt from the
# row-count check that flags stale arrays.
NON_PAIR_ARRAYS = {"clip_image_align"}


# ---------------------------------------------------------------------------
# Cache validity
# ---------------------------------------------------------------------------
def subset_fingerprint(captions, n_images=None, captions_per_image=None, seed=None):
    """A short hash of the exact caption list plus the settings that shape it.

    Cached arrays are reused only when this matches, so changing `N_IMAGES`, the
    cleaning rules or `SEED` throws the cache out automatically. Without it, a
    re-run could quietly pair new captions with old vectors.
    """
    n_images = cfg.N_IMAGES if n_images is None else n_images
    captions_per_image = cfg.CAPTIONS_PER_IMAGE if captions_per_image is None else captions_per_image
    seed = cfg.SEED if seed is None else seed

    src = "\n".join(captions) + f"|{n_images}|{captions_per_image}|{seed}"
    return hashlib.sha256(src.encode("utf-8")).hexdigest()[:16]


def cache_is_usable(fingerprint, previous_metadata=None, reuse=None, force_recompute=None):
    """Whether cached embeddings may be reused for this run."""
    reuse = cfg.REUSE_EXISTING if reuse is None else reuse
    force_recompute = cfg.FORCE_RECOMPUTE if force_recompute is None else force_recompute
    previous = (previous_metadata or {}).get("subset_fingerprint")
    return bool(reuse and not force_recompute and previous == fingerprint)


# ---------------------------------------------------------------------------
# Embedding arrays
# ---------------------------------------------------------------------------
def embedding_path(name, emb_dir=None):
    return (cfg.EMB_DIR if emb_dir is None else Path(emb_dir)) / f"{name}.npy"


def save_embedding(name, array, emb_dir=None):
    path = embedding_path(name, emb_dir)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, array)
    return path


def load_embedding(name, emb_dir=None, mmap=False):
    return np.load(embedding_path(name, emb_dir), mmap_mode="r" if mmap else None)


def load_or_encode(names, encode_fn, n_rows, cache_usable=False, save=True,
                   reuse_log=None, emb_dir=None):
    """Return the named arrays, loading them from disk when a valid cache exists.

    `encode_fn` is only called on a miss, and its results are saved immediately so
    that a failure further down never throws away encoding work.

    Returns `(*arrays, elapsed_seconds)`; elapsed is 0.0 on a cache hit.
    """
    reuse_log = {} if reuse_log is None else reuse_log
    paths = [embedding_path(n, emb_dir) for n in names]

    if cache_usable and all(p.is_file() for p in paths):
        arrays = [np.load(p) for p in paths]
        if all(len(a) == n_rows for a in arrays):
            print(f"Reusing cached: {', '.join(p.name for p in paths)}")
            reuse_log.update({n: "reused" for n in names})
            return (*arrays, 0.0)
        print("  cached arrays have the wrong row count -- recomputing")

    t0 = time.time()
    result = encode_fn()
    arrays = list(result) if isinstance(result, tuple) else [result]
    elapsed = time.time() - t0

    for name, array in zip(names, arrays):
        if save:
            save_embedding(name, array, emb_dir)
            reuse_log[name] = "computed"
        else:
            reuse_log[name] = "computed (not saved)"
    return (*arrays, elapsed)


def save_arrays(arrays, manifest, emb_dir=None):
    """Write every array, each row-checked against the manifest first.

    Returns a list of `(name, shape, dtype, megabytes)` for the notebook to print.
    """
    rows = []
    for name, array in arrays.items():
        assert len(array) == len(manifest), (
            f"{name} has {len(array)} rows but the manifest has {len(manifest)}"
        )
        save_embedding(name, array, emb_dir)
        rows.append((name, array.shape, str(array.dtype), array.nbytes / 1024 ** 2))
    return rows


def find_stale(arrays, n_rows, emb_dir=None):
    """Arrays on disk left over from an earlier run with a different subset size.

    Reported, never deleted -- throwing away someone else's work on a path guess
    is not this function's call to make.
    """
    emb_dir = cfg.EMB_DIR if emb_dir is None else Path(emb_dir)
    stale = []
    for p in sorted(emb_dir.glob("*.npy")):
        if p.stem in arrays or p.stem in NON_PAIR_ARRAYS:
            continue
        shape = np.load(p, mmap_mode="r").shape
        if shape[0] != n_rows:
            stale.append((rel(p), shape, p.stat().st_size / 1024 ** 2))
    return stale


def existing_embeddings(emb_dir=None):
    """What is already on disk, for the "what will this run cost" report."""
    emb_dir = cfg.EMB_DIR if emb_dir is None else Path(emb_dir)
    if not emb_dir.is_dir():
        return []
    return [
        (p.name, np.load(p, mmap_mode="r").shape, p.stat().st_size / 1024 ** 2)
        for p in sorted(emb_dir.glob("*.npy"))
    ]


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
def write_manifest(subset, parquet_path=None, csv_path=None):
    """Write the manifest in the exact row order of the embedding arrays.

    Image paths are stored relative to the project root, so the artefacts stay
    portable across machines.
    """
    parquet_path = cfg.MANIFEST_PARQUET if parquet_path is None else Path(parquet_path)
    csv_path = cfg.MANIFEST_CSV if csv_path is None else Path(csv_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    manifest = subset[MANIFEST_COLUMNS].copy()
    for col in ("image_path", "image_512_path"):
        manifest[col] = manifest[col].map(rel)

    manifest.to_parquet(parquet_path, index=False)
    manifest.to_csv(csv_path, index=False, encoding="utf-8")
    return manifest


def read_manifest(parquet_path=None):
    return pd.read_parquet(cfg.MANIFEST_PARQUET if parquet_path is None else parquet_path)


# ---------------------------------------------------------------------------
# Metadata
# ---------------------------------------------------------------------------
def load_metadata(path=None):
    """The previous run's metadata, or None if there is none."""
    path = cfg.METADATA_FILE if path is None else Path(path)
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def save_metadata(metadata, path=None):
    path = cfg.METADATA_FILE if path is None else Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    return path


def build_metadata(manifest, arrays, dataset_counts, cleaning_steps, timings,
                   fingerprint, reuse_log, not_persisted=None):
    """Assemble the provenance record.

    The config-derived sections are read straight from `config.py`; the caller
    supplies only the numbers that come out of the run itself.
    """
    import diffusers
    import torch
    import transformers

    return {
        "dataset": {
            "name": "COCO 2017 Captions",
            "split": "val2017",
            "annotations_file": rel(cfg.ANN_FILE),
            **dataset_counts,
            "n_curated_pairs": int(len(manifest)),
            "n_distinct_images": int(manifest["image_id"].nunique()),
            "captions_per_image": cfg.CAPTIONS_PER_IMAGE,
        },
        "image_preprocessing": {
            "size": [cfg.IMAGE_SIZE, cfg.IMAGE_SIZE],
            "mode": "RGB",
            "resize": "aspect-preserving shorter-side resize + centre crop (PIL ImageOps.fit, bicubic)",
            "exif_transpose": True,
            "normalisations": {"display_and_metrics": "[0, 1]", "sd_vae": "[-1, 1] via Normalize(0.5, 0.5)"},
            "cache_dir": rel(cfg.IMAGES_512_DIR),
        },
        "caption_cleaning": {
            "steps": list(cleaning_steps),
            "filters": {
                "min_words": cfg.MIN_WORDS, "max_words": cfg.MAX_WORDS,
                "min_chars": cfg.MIN_CHARS, "max_chars": cfg.MAX_CHARS,
                "drop_empty": True, "drop_duplicates": True, "drop_missing_image": True,
            },
            "stopwords": "not removed -- see the note in section 4",
        },
        "embeddings": {n: {"shape": list(a.shape), "dtype": str(a.dtype)} for n, a in arrays.items()},
        "embeddings_not_persisted": not_persisted or {},
        "models": {
            "clip_joint": cfg.CLIP_JOINT_MODEL,
            "clip_sd_text_encoder": cfg.CLIP_SD_MODEL,
            "bert": cfg.BERT_MODEL,
            "clip_max_tokens": cfg.CLIP_MAX_TOKENS,
            "bert_max_tokens": cfg.BERT_MAX_TOKENS,
            "encoder_dtype": str(cfg.ENCODER_DTYPE),
        },
        "encode_seconds": {k: round(v, 1) for k, v in timings.items()},
        "versions": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "diffusers": diffusers.__version__,
        },
        "seed": cfg.SEED,
        "subset_fingerprint": fingerprint,   # guards cache reuse on the next run
        "cache_status": dict(reuse_log),
    }


# ---------------------------------------------------------------------------
# Round trip
# ---------------------------------------------------------------------------
def verify_round_trip(manifest, arrays, check=("clip_text_joint", "bert_mean")):
    """Reload from disk and confirm it matches what is in memory.

    Without this, a silent write failure would only surface in a later notebook.
    Returns a small summary dict; raises AssertionError if anything disagrees.
    """
    reloaded = read_manifest()
    assert len(reloaded) == len(manifest), "manifest row count changed on the round trip"
    assert reloaded["pair_idx"].tolist() == list(range(len(reloaded))), "pair_idx is not contiguous"

    for name in check:
        on_disk = load_embedding(name)
        assert np.array_equal(on_disk, arrays[name]), f"{name} does not match what was written"

    probe = reloaded.iloc[min(7, len(reloaded) - 1)]
    return {
        "manifest_rows": int(len(reloaded)),
        "arrays_checked": list(check),
        "probe_caption": probe["caption"],
        "probe_image": probe["image_512_path"],
    }

In [ ]:
import sys

sys.path.insert(0, ".")
for _name in [m for m in sys.modules if m in ("config", "utils") or m.startswith("modules")]:
    del sys.modules[_name]          # so a re-run picks up the freshly written files

import config as cfg
import utils
from modules import arms, clip_embeddings, generation, prompts, recording, sanity_checks, sd_pipeline, storage

print("modules ready:", 8)

---
## 0. Paths on a hosted runtime

**Does nothing on a normal checkout**, so this notebook still runs unchanged locally.

It matters because the repository is **code only** — `data/` is git-ignored. Milestone 2 needs
milestone 1's `data/processed/` (manifest, embeddings and the 512px photo cache, ~390 MB), and a
`git clone` will not bring that with it.

There are two ways to supply it, and this cell handles both:

- **Attach it as a Kaggle Dataset.** The cell finds whichever attached dataset holds
  `processed/pairs.parquet`, copies it to `/kaggle/working/data` and points `config.py` there. The
  copy is not optional — `/kaggle/input` is read-only and `cfg.ensure_dirs()` creates directories
  under `DATA_DIR`.
- **Clone the repo somewhere writable and run milestone 1 first.** Then nothing needs redirecting
  and this cell stays out of the way.

On Colab `/content` is writable, so the second case applies and this cell is a no-op there.

This notebook also pulls ~7 GB of aMUSEd and LDM-BERT weights, so it needs the internet toggle
**on**, or those weights attached as a second dataset with `HF_HOME` pointed at it.

It must stay **above the import cell** — `config.py` resolves every path once, when it is first
imported, and setting the variables afterwards has no effect.

In [ ]:
import time, warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

import config as cfg
import utils
from modules import arms, clip_embeddings, prompts, recording, sanity_checks, sd_pipeline, storage

cfg.ensure_dirs()
utils.set_seed()

manifest = storage.read_manifest()
picks = prompts.fixed_prompts(manifest)
PROMPTS = picks["caption"].tolist()
REAL_IMAGES = [cfg.PROJECT_ROOT / p for p in picks["image_512_path"]]

clip_tokenizer, clip_joint, clip_processor = clip_embeddings.load_joint()

print(f"{len(PROMPTS)} prompts, seed {cfg.SEED}")
print(f"start: {sd_pipeline.vram_report()}")

## 1. SD 1.5 — the reference arm

The baseline from the sweep notebook, at DPMSolver++ and 20 steps.

In [ ]:
panels, timings, notes = {}, {}, {}

pipe, repo, base_config = sd_pipeline.load_sd()
sd_pipeline.set_scheduler(pipe, "dpmpp", base_config=base_config)

t0 = time.time()
panels["SD 1.5 (CLIP, diffusion, 512)"] = arms.run_arm(
    pipe, PROMPTS, num_inference_steps=cfg.SAMPLERS["dpmpp"], guidance_scale=cfg.GUIDANCE,
    generator=torch.Generator(device="cpu").manual_seed(cfg.SEED))
timings["SD 1.5 (CLIP, diffusion, 512)"] = time.time() - t0
notes["SD 1.5 (CLIP, diffusion, 512)"] = f"{repo}, DPMSolver++ {cfg.SAMPLERS['dpmpp']} steps"

print(f"{timings['SD 1.5 (CLIP, diffusion, 512)']:.1f}s   peak {sd_pipeline.vram_report()['peak_gb']} GB")

## 2. aMUSEd — the same encoder, a different mechanism

No `num_inference_steps` in the diffusion sense: aMUSEd unmasks tokens over a fixed number of rounds.
`arms.run_arm` passes only the arguments this pipeline actually declares, so nothing from SD's
vocabulary leaks across.

In [ ]:
amused, spec = arms.load_arm("amused", previous=pipe)
del pipe

t0 = time.time()
label = f"aMUSEd (CLIP, masked transformer, {spec['size']})"
panels[label] = arms.run_arm(
    amused, PROMPTS, num_inference_steps=12, guidance_scale=cfg.GUIDANCE,
    generator=torch.Generator(device="cpu").manual_seed(cfg.SEED))
timings[label] = time.time() - t0
notes[label] = f"{spec['repo']}, 12 unmasking rounds, encoder: {spec['encoder']}"

print(f"{timings[label]:.1f}s   peak {sd_pipeline.vram_report()['peak_gb']} GB")

## 3. LDM-BERT — the BERT-conditioned arm

Latent diffusion like SD, but conditioned on BERT. This is milestone 2's BERT coverage without
training anything: a pretrained model that already does what a BERT→CLIP adapter would have to learn.

In [ ]:
ldm, spec = arms.load_arm("ldm_bert", previous=amused)
del amused

t0 = time.time()
label = f"LDM-BERT (BERT, diffusion, {spec['size']})"
panels[label] = arms.run_arm(
    ldm, PROMPTS, num_inference_steps=50, guidance_scale=cfg.GUIDANCE,
    generator=torch.Generator(device="cpu").manual_seed(cfg.SEED))
timings[label] = time.time() - t0
notes[label] = f"{spec['repo']}, 50 steps, encoder: {spec['encoder']}"

sd_pipeline.free(ldm); del ldm
print(f"{timings[label]:.1f}s   released: {sd_pipeline.vram_report()}")

## 4. Scoring

One metric across all three: CLIP(text, generated), the same cosine the sweep used. It is scored in
CLIP ViT-B/32's joint space — a *different* CLIP from the ViT-L/14 that conditions SD and aMUSEd, so
the judge is not simply one of the contestants' own encoders.

Resolution does not bias it: CLIP resizes everything to 224 before looking, so the 256px arms are not
penalised for being small. What a bare cosine *does* miss is image quality — a sharp image and a
muddy one that both contain the right objects score alike. Read the grid alongside the table.

In [ ]:
rows = []
for label, images in panels.items():
    txt_gen, txt_real, gen_real = sanity_checks.clip_scores(
        PROMPTS, images, REAL_IMAGES, clip_tokenizer, clip_joint, clip_processor)
    rows.append({
        "arm": label,
        "clip_text_generated": round(float(txt_gen.mean()), 4),
        "std": round(float(txt_gen.std()), 4),
        "clip_generated_real": round(float(gen_real.mean()), 4),
        "seconds": round(timings[label], 1),
        "s_per_image": round(timings[label] / len(PROMPTS), 2),
        "size": images[0].size[0],
    })

results = pd.DataFrame(rows).sort_values("clip_text_generated", ascending=False)
reference = float(txt_real.mean())
print(f"real COCO photographs score {reference:.4f} -- the number to read these against\n")
display(results)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.0))
y = np.arange(len(results))
ax.barh(y, results["clip_text_generated"], xerr=results["std"], capsize=4)
ax.axvline(reference, ls="--", c="grey", label=f"real photos ({reference:.3f})")
ax.set_yticks(y); ax.set_yticklabels(results["arm"], fontsize=8); ax.invert_yaxis()
ax.set_xlabel("CLIP(text, generated)")
ax.set_title("Prompt adherence across generative mechanisms")
ax.legend(); ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
print(utils.save_fig(fig, "m2_arm_comparison"))
plt.show()

## 5. The pictures

One row per arm, the same prompts down the columns. Tiles are resized to a common width so the rows
line up — the 256px arms genuinely produce smaller images, which the table records.

In [ ]:
WIDTH = 256
normalised = {label: [img.resize((WIDTH, WIDTH)) for img in images]
              for label, images in panels.items()}

grid = recording.comparison_grid(normalised)
grid_path = cfg.SWEEP_DIR / "arm_comparison.png"
grid.save(grid_path)
print(utils.rel(grid_path))

for label, images in panels.items():
    print(recording.save_run(images, PROMPTS, recording.slugify(label))[-1])

grid

In [ ]:
summary = {
    "prompts": PROMPTS,
    "seed": cfg.SEED,
    "protocol_note": ("shared prompts and seed, not shared latents -- aMUSEd has no starting "
                      "noise, so the sweep's fixed-latent protocol does not transfer"),
    "scoring_note": "CLIP ViT-B/32 judges; SD and aMUSEd are conditioned on ViT-L/14",
    "reference_real_photos": round(reference, 4),
    "arms": {row["arm"]: {**{k: v for k, v in row.items() if k != "arm"},
                          "detail": notes[row["arm"]]} for row in rows},
}
print(utils.save_table("milestone_2_comparison", summary))
print()
print(f"best adherence: {results.iloc[0]['arm']}  ({results.iloc[0]['clip_text_generated']:.4f})")
print(f"fastest       : {results.sort_values('s_per_image').iloc[0]['arm']}  "
      f"({results['s_per_image'].min():.2f}s per image)")

### What this shows

The sweep notebook spent its time on samplers and noise schedules. aMUSEd has none of those, and still
turns captions into pictures from the same CLIP embedding — so the noise schedule is a property of
*one way* of generating images, not of text-to-image generation as such.

LDM-BERT swaps the encoder instead of the mechanism, which is the other half of milestone 2's BERT
coverage and needs no training to get.